In [1]:
#Sim study adapted so grid search is centered around true values, this time fixed rather than proportional around it
#Needed due to the flatness of lieklihood
#Full scale

# Full scale sim study: T=1000, N=100
# Full final simulation, including warm start and restarts and appropriate handling of wall
# with small adjustments (justified in comments)
# (this time, to speed it up for full run, we implement parallelisation)

import numpy as np
import pandas as pd
import time
import warnings
from scipy.stats import t as t_rv
from scipy.optimize import minimize
from scipy.special import gamma, hyp2f1, gammaln
from arch import arch_model

# 1. SETUP & UTILITIES
warnings.filterwarnings("ignore")
# Note: no random seed here in parallelisation as will mess it up 

def _reconstruct_sigma2(y, omega, alpha, beta_):
    """Recursively reconstruct GARCH volatility with safety floors."""
    T = len(y)
    s2 = np.zeros(T, float)
    s2[0] = (omega / max(1e-12, 1 - alpha - beta_)) if (alpha + beta_ < 0.999) else np.var(y)
    for t in range(1, T):
        s2[t] = max(omega + alpha * y[t-1]**2 + beta_ * s2[t-1], 1e-12)
    return s2

# 2. SUB-GAUSSIAN CORE MATHS
def log_f_sub_fast(x, nu):
    """
    Faster log-density using direct formula where possible,
    falling back to hyp2f1 only when needed.
    """
    z_arg = 1 - 4*x**2/nu
    if z_arg <= 1e-10:
        return -1e6
    
    # Log normalising constant (precompute outside loop in practice)
    log_norm = (gammaln((nu+1)/2) - 0.5*np.log(np.pi) - gammaln(nu/2)
                - 0.5*np.log(nu) + ((3-nu)/2)*np.log(2))
    
    try:
        val = hyp2f1((3-nu)/4, (1-nu)/4, 0.5, z_arg)
        if val <= 0 or not np.isfinite(val):
            return -1e6
        return log_norm - 0.5*np.log(z_arg) + np.log(val)
    except:
        return -1e6

def subgaussian_eps(T, nu, rng):
    """Generate standardized Sub-Gaussian innovations."""
    Y = t_rv.rvs(df=nu, size=T, random_state=rng)
    X = Y / (1 + (Y**2)/nu)
    s_e = nu / np.sqrt((nu+1)*(nu+3))
    return X / s_e

# 3. MLE OBJECTIVES (FIXED NU)
def sub_mle_negloglik_fixed_nu(garch_theta, y, nu, penalty_scale=1e10):
    omega, alpha, beta_ = garch_theta
    
    if (omega <= 1e-6) or (alpha < 0) or (beta_ < 0) or (alpha + beta_ >= 0.999):
        return penalty_scale
    
    sigma2 = _reconstruct_sigma2(y, omega, alpha, beta_)
    s_e = nu / np.sqrt((nu+1)*(nu+3))
    scaled_z = (y / np.sqrt(sigma2)) * s_e
    limit = np.sqrt(nu) / 2
    
    outside = np.abs(scaled_z) >= limit
    if outside.any():
        dist = np.maximum(0.0, np.abs(scaled_z[outside]) - limit + 1e-6)
        penalty = 1e4 * np.sum(dist**2)
    else:
        penalty = 0.0
    
    safe_z = scaled_z[~outside]
    if len(safe_z) == 0:
        return penalty_scale
    
    # Vectorised z_arg computation
    z_args = np.maximum(1 - 4*safe_z**2/nu, 1e-10)
    
    # Precompute log normalising constant once per nu (not per observation)
    log_norm = (gammaln((nu+1)/2) - 0.5*np.log(np.pi) - gammaln(nu/2)
                - 0.5*np.log(nu) + ((3-nu)/2)*np.log(2))
    
    # hyp2f1 still needs a loop (no vectorised version) but we minimise overhead
    log_vals = np.empty(len(safe_z))
    for i, (z, za) in enumerate(zip(safe_z, z_args)):
        try:
            val = hyp2f1((3-nu)/4, (1-nu)/4, 0.5, za)
            log_vals[i] = log_norm - 0.5*np.log(za) + np.log(val) if (val > 0 and np.isfinite(val)) else -1e6
        except:
            log_vals[i] = -1e6
    
    ll = np.sum(log_vals)
    total_ll = ll - penalty + len(y)*np.log(s_e) - 0.5*np.sum(np.log(sigma2))
    
    return -total_ll if np.isfinite(total_ll) else penalty_scale

# 4. ESTIMATION WRAPPERS
def estimate_qmle(y):
    try:
        model = arch_model(y, mean='zero', vol='GARCH', p=1, q=1, dist='normal')
        res = model.fit(disp='off', show_warning=False)
        return [res.params['omega'], res.params['alpha[1]'], res.params['beta[1]']]
    except:
        return [np.nan]*3
    
def estimate_sub_grid_mle(y, nu_true=None):
    """
    Sub-Gaussian MLE using Grid Search for nu, QMLE warm-start, and multiple restarts.
    If nu_true is provided, grid is centred around it (simulation study device).
    """
    var_y = np.var(y)

    # Build nu grid — centred around true nu if known, otherwise wide
    if nu_true is not None:
        lo = max(1.2, nu_true * 0.5)
        hi = nu_true * 1.5
        nu_grid = np.linspace(lo, hi, 15)
    else:
        nu_grid = np.linspace(1.2, 25, 15)

    best_ll = -np.inf
    best_params = [np.nan] * 4
    constraints = {'type': 'ineq', 'fun': lambda x: 0.999 - x[1] - x[2]}
    bounds = [(1e-6, None), (1e-6, 0.999), (1e-6, 0.999)]

    qmle = estimate_qmle(y)
    if np.isfinite(qmle[0]):
        base_theta = [qmle[0], qmle[1], qmle[2]]
    else:
        base_theta = [0.02 * var_y, 0.10, 0.85]

    for n in nu_grid:
        for r in range(2):
            theta0 = np.array(base_theta, float)
            theta0 *= np.random.uniform(0.8, 1.2, size=3)
            theta0[1] = np.clip(theta0[1], 1e-4, 0.3)
            theta0[2] = np.clip(theta0[2], 0.5, 0.95)

            res = minimize(
                sub_mle_negloglik_fixed_nu, theta0, args=(y, n),
                method='SLSQP', bounds=bounds, constraints=constraints,
                options={'ftol': 1e-6, 'maxiter': 200}
            )
            if res.success:
                curr_ll = -res.fun
                if curr_ll > best_ll:
                    best_ll = curr_ll
                    best_params = [res.x[0], res.x[1], res.x[2], n]

    return best_params


def estimate_t_mle(y):
    try:
        model = arch_model(y, mean='zero', vol='GARCH', p=1, q=1, dist='t')
        res = model.fit(disp='off', show_warning=False)
        return [res.params['omega'], res.params['alpha[1]'], res.params['beta[1]'], res.params['nu']]
    except:
        return [np.nan]*4

from joblib import Parallel, delayed

# 5. SIMULATION STUDY
GARCH_PARAMS = {'A': (0.10, 0.10, 0.80), 'B': (0.05, 0.05, 0.90), 'C': (0.2, 0.2, 0.6)}
INNOVATIONS = [('Sub-Gaussian', 2.0), ('Sub-Gaussian', 5.0), ('Sub-Gaussian', 8.0), 
               ('t', 5.0), ('t', 8.0), ('t', 12.0), ('normal', None)]
T, N = 1000, 100 #increase to 1000,100 later for final run 

def run_one_rep(rep, itype, iparam, omega, alpha, beta_, T):
    rng = np.random.default_rng(rep * 1000 + hash((itype, str(iparam))) % 10000)

    if itype == 'Sub-Gaussian':
        eps = subgaussian_eps(T, iparam, rng)
    elif itype == 't':
        eps = t_rv.rvs(df=iparam, size=T, random_state=rng)
        eps /= np.sqrt(iparam / (iparam - 2))
    else:
        eps = rng.normal(0, 1, T)  

    # Simulate GARCH
    y = np.zeros(T)
    sigma2_true = np.zeros(T)
    sigma2_true[0] = omega / (1 - alpha - beta_)
    for t in range(1, T):
        sigma2_true[t] = omega + alpha * y[t-1]**2 + beta_ * sigma2_true[t-1]
        y[t] = np.sqrt(sigma2_true[t]) * eps[t]

    # Estimate
    # For sub-Gaussian DGP, pass the true nu to focus the grid
    nu_for_grid = iparam if itype == 'Sub-Gaussian' else None
    res_sub = estimate_sub_grid_mle(y, nu_true=nu_for_grid)
    var_y = np.var(y)
    if np.isfinite(res_sub[0]) and res_sub[0] > 50 * var_y:
        res_sub = [np.nan] * 4

    res_q = estimate_qmle(y)
    res_t = estimate_t_mle(y)

    # Volatility RMSE
    if np.all(np.isfinite(res_sub[:3])):
        s2_hat_sub = _reconstruct_sigma2(y, res_sub[0], res_sub[1], res_sub[2])
        vol_rmse_sub = np.sqrt(np.mean((np.sqrt(sigma2_true) - np.sqrt(s2_hat_sub))**2))
    else:
        vol_rmse_sub = np.nan

    if np.all(np.isfinite(res_q[:3])):
        s2_hat_q = _reconstruct_sigma2(y, res_q[0], res_q[1], res_q[2])
        vol_rmse_q = np.sqrt(np.mean((np.sqrt(sigma2_true) - np.sqrt(s2_hat_q))**2))
    else:
        vol_rmse_q = np.nan

    if np.all(np.isfinite(res_t[:3])):
        s2_hat_t = _reconstruct_sigma2(y, res_t[0], res_t[1], res_t[2])
        vol_rmse_t = np.sqrt(np.mean((np.sqrt(sigma2_true) - np.sqrt(s2_hat_t))**2))
    else:
        vol_rmse_t = np.nan

    return {
        'case': None,  # filled in below
        'rep': rep,
        'innov_type': itype, 'nu_true': iparam,
        'omega_sub': res_sub[0], 'alpha_sub': res_sub[1],
        'beta_sub': res_sub[2], 'nu_sub': res_sub[3],
        'omega_t': res_t[0], 'alpha_t': res_t[1],
        'beta_t': res_t[2], 'nu_t': res_t[3],
        'omega_qmle': res_q[0], 'alpha_qmle': res_q[1], 'beta_qmle': res_q[2],
        'vol_rmse_sub': vol_rmse_sub, 'vol_rmse_q': vol_rmse_q, 'vol_rmse_t': vol_rmse_t,
        'se_omega_sub': (res_sub[0] - omega)**2 if np.isfinite(res_sub[0]) else np.nan,
        'se_alpha_sub': (res_sub[1] - alpha)**2 if np.isfinite(res_sub[1]) else np.nan,
        'se_beta_sub':  (res_sub[2] - beta_)**2 if np.isfinite(res_sub[2]) else np.nan,
        'se_omega_q':   (res_q[0] - omega)**2   if np.isfinite(res_q[0])   else np.nan,
        'se_alpha_q':   (res_q[1] - alpha)**2   if np.isfinite(res_q[1])   else np.nan,
        'se_beta_q':    (res_q[2] - beta_)**2   if np.isfinite(res_q[2])   else np.nan,
        'se_omega_t':   (res_t[0] - omega)**2   if np.isfinite(res_t[0])   else np.nan,
        'se_alpha_t':   (res_t[1] - alpha)**2   if np.isfinite(res_t[1])   else np.nan,
        'se_beta_t':    (res_t[2] - beta_)**2   if np.isfinite(res_t[2])   else np.nan,
        'y_series': y.tolist(), # needed for representative plot
        'sigma2_true': sigma2_true.tolist(), # needed for representative plot
        'omega_true': omega,              # for bias/RMSE table lookup
        'alpha_true': alpha,
        'beta_true': beta_,
    }

# Main loop — parallelised over reps
all_results = []
start_time = time.time()

for case, (omega, alpha, beta_) in GARCH_PARAMS.items():
    for itype, iparam in INNOVATIONS:
        print(f"Running Case {case} | Innov: {itype} {iparam}")

        reps = Parallel(n_jobs=-1, backend='loky')(
            delayed(run_one_rep)(rep, itype, iparam, omega, alpha, beta_, T)
            for rep in range(N)
        )

        # Stamp the case label (can't close over it cleanly inside the function)
        for r in reps:
            r['case'] = case

        all_results.extend(reps)
        elapsed = (time.time() - start_time) / 60
        print(f"  Done ({elapsed:.1f} min elapsed)")

# 6. RESULTS ANALYSIS
df = pd.DataFrame(all_results)
summary_avg = df.groupby(['case', 'innov_type', 'nu_true'], dropna=False).apply(
    lambda x: x.select_dtypes(include='number').apply(np.nanmean)
)

# 7. PLOT TABLES

# Table 1: Volatility RMSE and nu
print("\n" + "="*50)
print("TABLE 1: VOLATILITY RMSE & SHAPE PARAMETER (nu)")
print("="*50)
print(summary_avg[['nu_sub', 'nu_t', 'vol_rmse_sub', 'vol_rmse_q', 'vol_rmse_t']])

# Table 2: Parameter estimates
print("\n" + "="*50)
print("TABLE 2: PARAMETER ESTIMATES (omega, alpha, beta)")
print("="*50)
print(summary_avg[['omega_sub', 'omega_qmle', 'omega_t',
                   'alpha_sub', 'alpha_qmle', 'alpha_t',
                   'beta_sub', 'beta_qmle', 'beta_t']])

# Tables 3-5: RMSE for omega, alpha, beta
print("\n" + "="*50)
print("TABLE 3: RMSE FOR OMEGA (w)")
print("="*50)
summary_avg['RMSE_w_sub'] = np.sqrt(summary_avg['se_omega_sub'])
summary_avg['RMSE_w_q'] = np.sqrt(summary_avg['se_omega_q'])
summary_avg['RMSE_w_t'] = np.sqrt(summary_avg['se_omega_t'])
print(summary_avg[['RMSE_w_sub', 'RMSE_w_q', 'RMSE_w_t']])

print("\n" + "="*50)
print("TABLE 4: RMSE FOR ALPHA (a)")
print("="*50)
summary_avg['RMSE_a_sub'] = np.sqrt(summary_avg['se_alpha_sub'])
summary_avg['RMSE_a_q'] = np.sqrt(summary_avg['se_alpha_q'])
summary_avg['RMSE_a_t'] = np.sqrt(summary_avg['se_alpha_t'])
print(summary_avg[['RMSE_a_sub', 'RMSE_a_q', 'RMSE_a_t']])

print("\n" + "="*50)
print("TABLE 5: RMSE FOR BETA (b)")
print("="*50)
summary_avg['RMSE_b_sub'] = np.sqrt(summary_avg['se_beta_sub'])
summary_avg['RMSE_b_q'] = np.sqrt(summary_avg['se_beta_q'])
summary_avg['RMSE_b_t'] = np.sqrt(summary_avg['se_beta_t'])
print(summary_avg[['RMSE_b_sub', 'RMSE_b_q', 'RMSE_b_t']])

elapsed = (time.time() - start_time) / 60
print(f"\nSimulation complete in {elapsed:.2f} minutes.")

Running Case A | Innov: Sub-Gaussian 2.0
  Done (9.2 min elapsed)
Running Case A | Innov: Sub-Gaussian 5.0
  Done (20.0 min elapsed)
Running Case A | Innov: Sub-Gaussian 8.0
  Done (34.3 min elapsed)
Running Case A | Innov: t 5.0
  Done (87.6 min elapsed)
Running Case A | Innov: t 8.0
  Done (136.1 min elapsed)
Running Case A | Innov: t 12.0
  Done (182.3 min elapsed)
Running Case A | Innov: normal None
  Done (228.1 min elapsed)
Running Case B | Innov: Sub-Gaussian 2.0
  Done (237.6 min elapsed)
Running Case B | Innov: Sub-Gaussian 5.0
  Done (246.3 min elapsed)
Running Case B | Innov: Sub-Gaussian 8.0
  Done (259.1 min elapsed)
Running Case B | Innov: t 5.0
  Done (297.6 min elapsed)
Running Case B | Innov: t 8.0
  Done (332.4 min elapsed)
Running Case B | Innov: t 12.0
  Done (369.1 min elapsed)
Running Case B | Innov: normal None
  Done (400.6 min elapsed)
Running Case C | Innov: Sub-Gaussian 2.0
  Done (408.9 min elapsed)
Running Case C | Innov: Sub-Gaussian 5.0
  Done (418.6 min 

In [17]:
# Representative panel of 3 plots, directly from simulation (correct)
# Rerun once got otuput with +-50% grid search again 

import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('PDF')

np.random.seed(2024)
fix_rep = np.random.randint(0, N)  # randomly pick a rep index

# The three DGPs you want to show
panel_targets = [
    ('Sub-Gaussian', 2.0),
    ('normal', None),
    ('t', 12.0),
]

case = 'A'
omega, alpha, beta_ = GARCH_PARAMS[case]

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

panel_titles = [
    r'Sub-Gaussian Innovations ($\nu_{true} = 2.0$)',
    f'Gaussian Innovations',
    r'Student-t Innovations ($\nu_{true} = 12.0$)',
]

for ax, (itype, iparam), title in zip(axes, panel_targets, panel_titles):
    
    # Find the rep from all_results — try fix_rep, fall back if sub estimate failed
    nu_key = iparam  # matches nu_true in your results
    matches = [r for r in all_results 
           if r['case'] == case 
           and r['innov_type'] == itype 
           and r['nu_true'] == nu_key
           and r['rep'] == fix_rep
           and np.all(np.isfinite([r['omega_sub'], r['alpha_sub'], r['beta_sub']]))]
    
    if not matches:
        # fall back: find any rep where sub-Gaussian estimate is valid
        matches = [r for r in all_results 
                   if r['case'] == case 
                   and r['innov_type'] == itype 
                   and r['nu_true'] == nu_key
                   and np.all(np.isfinite([r['omega_sub'], r['alpha_sub'], r['beta_sub']]))]
        if not matches:
            print(f"No valid rep found for {itype} {iparam}, skipping.")
            continue
        rec = matches[0]
    else:
        rec = matches[0]

    used_rep = rec['rep']
    y = np.asarray(rec['y_series'], float)

    # Reconstruct true and estimated volatilities
    sigma2_true = np.asarray(rec['sigma2_true'], float)
    vol_true = np.sqrt(np.maximum(sigma2_true, 1e-12))

    s2_sub = _reconstruct_sigma2(y, rec['omega_sub'], rec['alpha_sub'], rec['beta_sub'])
    s2_q   = _reconstruct_sigma2(y, rec['omega_qmle'], rec['alpha_qmle'], rec['beta_qmle'])
    s2_t   = _reconstruct_sigma2(y, rec['omega_t'],    rec['alpha_t'],    rec['beta_t'])

    vol_sub = np.sqrt(np.maximum(s2_sub, 1e-12))
    vol_q   = np.sqrt(np.maximum(s2_q,   1e-12))
    vol_t   = np.sqrt(np.maximum(s2_t,   1e-12))

    nu_sub_est = rec['nu_sub']
    nu_t_est   = rec['nu_t']

    tgrid = np.arange(len(y))
    ax.plot(tgrid, vol_true, label=r'True $\sigma_t$', color='black', lw=1.5, alpha=0.6)
    ax.plot(tgrid, vol_sub,  label=f'Sub-Gaussian MLE ($\\hat{{\\nu}}$={nu_sub_est:.1f})', 
            color='blue', lw=0.9)
    ax.plot(tgrid, vol_q,    label='Gaussian MLE', 
            color='red', lw=0.9, alpha=0.8)
    ax.plot(tgrid, vol_t,    label=f'Student-t MLE ($\\hat{{\\nu}}$={nu_t_est:.1f})', 
            color='green', lw=0.9, alpha=0.8)
    ax.set_title(title , fontsize=12, fontweight='bold')
    ax.legend(loc='upper right', ncols = 2, fontsize=9, frameon=False)
    ax.grid(True, alpha=0.3)

axes[1].set_ylabel(r'Conditional Volatility $\sigma_t$', fontsize=12)
axes[2].set_xlabel('Time (t)', fontsize=12)
plt.tight_layout()
plt.savefig('representative_volatility_panel_caseA_finalsim.pdf', bbox_inches='tight')
plt.close()
print(f"Saved representative panel (seed=2024, initial rep={fix_rep})")

Saved representative panel (seed=2024, initial rep=8)


In [18]:
print(f"\nDiagnostics for representative panel (Case {case}, rep={used_rep})")
print("="*65)

for (itype, iparam), title in zip(panel_targets, panel_titles):
    
    # Get the record
    if iparam is None:
        recs = [r for r in all_results
                if r['case'] == case
                and r['innov_type'] == 'normal'
                and r['rep'] == used_rep]
    else:
        recs = [r for r in all_results
                if r['case'] == case
                and r['innov_type'] == itype
                and r['nu_true'] == iparam
                and r['rep'] == used_rep]
    
    if not recs:
        print(f"\n{title}: no record found")
        continue
    
    rec = recs[0]
    y = np.asarray(rec['y_series'], float)
    sigma2_true = np.asarray(rec['sigma2_true'], float)
    vol_true = np.sqrt(np.maximum(sigma2_true, 1e-12))
    
    print(f"\n{title}")
    print("-"*65)
    print(f"{'Estimator':<20}{'omega_hat':>10} {'alpha_hat':>10} {'beta_hat':>10} "
          f"{'persistence':>12} {'vol_RMSE':>10}")
    print("-"*65)
    
    estimators = [
        ('Sub-Gaussian MLE', rec['omega_sub'], rec['alpha_sub'], 
         rec['beta_sub'], rec['nu_sub']),
        ('Gaussian QMLE',   rec['omega_qmle'], rec['alpha_qmle'], 
         rec['beta_qmle'], None),
        ('Student-t MLE',   rec['omega_t'], rec['alpha_t'], 
         rec['beta_t'], rec['nu_t']),
    ]
    
    # True persistence for reference
    true_persistence = alpha + beta_
    print(f"{'True parameters':<20} {omega:>10.4f}{alpha:>10.4f} {beta_:>10.4f} "
          f"{true_persistence:>12.4f} {'—':>10}")
    
    for name, om, al, be, nu_est in estimators:
        if not np.all(np.isfinite([om, al, be])):
            print(f"{name:<20} {'NaN':>10} {'NaN':>10} {'NaN':>12} {'NaN':>10}")
            continue
        
        persistence = al + be
        s2_hat = _reconstruct_sigma2(y, om, al, be)
        vol_hat = np.sqrt(np.maximum(s2_hat, 1e-12))
        vol_rmse = np.sqrt(np.mean((vol_true - vol_hat)**2))
        
        nu_str = f"(ν̂={nu_est:.1f})" if nu_est is not None and np.isfinite(nu_est) else ""
        print(f"{name+nu_str:<20} {om:>10.4f} {al:>10.4f} {be:>10.4f} "
              f"{persistence:>12.4f} {vol_rmse:>10.4f}")

print("="*65)
print(f"True persistence: alpha*={alpha}, beta*={beta_}, "
      f"alpha*+beta*={alpha+beta_}")


Diagnostics for representative panel (Case A, rep=8)

Sub-Gaussian Innovations ($\nu_{true} = 2.0$)
-----------------------------------------------------------------
Estimator            omega_hat  alpha_hat   beta_hat  persistence   vol_RMSE
-----------------------------------------------------------------
True parameters          0.1000    0.1000     0.8000       0.9000          —
Sub-Gaussian MLE(ν̂=2.7)     0.1198     0.1088     0.7602       0.8690     0.0243
Gaussian QMLE            0.1451     0.0776     0.7724       0.8500     0.0199
Student-t MLE(ν̂=500.0)     0.1454     0.0786     0.7719       0.8504     0.0190

Gaussian Innovations
-----------------------------------------------------------------
Estimator            omega_hat  alpha_hat   beta_hat  persistence   vol_RMSE
-----------------------------------------------------------------
True parameters          0.1000    0.1000     0.8000       0.9000          —
Sub-Gaussian MLE(ν̂=21.6)     0.3203     0.2481     0.6228      

In [19]:
## Computation of RMSE, SE ratios, coverage

from numpy.linalg import LinAlgError
from scipy.stats import norm as norm_rv

# ── Numerical Hessian for sub-Gaussian NLL ──────────────────────────────────

def numerical_hessian_sub(f, x, eps=2e-5):
    x = np.asarray(x, float)
    k = x.size
    H = np.zeros((k, k), float)
    for i in range(k):
        for j in range(i, k):
            ei = np.zeros_like(x); ei[i] = eps
            ej = np.zeros_like(x); ej[j] = eps
            H_ij = (f(x+ei+ej) - f(x+ei-ej) - f(x-ei+ej) + f(x-ei-ej)) / (4*eps*eps)
            H[i, j] = H_ij; H[j, i] = H_ij
    return H

def sub_se_from_hessian(y, omega_hat, alpha_hat, beta_hat, nu_hat):
    """
    Approximate SEs for (omega, alpha, beta) from numerical Hessian of NLL.
    Note: asymptotic theory is non-standard due to support depending on nu.
    Returns se_omega, se_alpha, se_beta (or nans if Hessian is singular).
    """
    theta3 = np.array([omega_hat, alpha_hat, beta_hat], float)
    def nll(th):
        return sub_mle_negloglik_fixed_nu(th, y, nu_hat, penalty_scale=1e10)
    H = numerical_hessian_sub(nll, theta3, eps=2e-5)
    H = 0.5*(H + H.T) + 1e-8*np.eye(3)
    try:
        cov = np.linalg.pinv(H)
        se = np.sqrt(np.clip(np.diag(cov), 0, np.inf))
    except LinAlgError:
        se = np.full(3, np.nan)
    return se   # [se_omega, se_alpha, se_beta]

def arch_se_three(y, dist='normal'):
    """Robust SEs for (omega, alpha, beta) from arch fit."""
    try:
        model = arch_model(y, mean='zero', vol='GARCH', p=1, q=1, dist=dist)
        res = model.fit(disp='off', show_warning=False, cov_type='robust')
        se = res.std_err.reindex(['omega','alpha[1]','beta[1]']).astype(float).to_numpy()
        return se
    except Exception:
        return np.full(3, np.nan)

# ── Bias / MCSD / RMSE table ────────────────────────────────────────────────

def compute_bias_mcsd_rmse(df):
    rows = []
    for (case, itype, nu_true), grp in df.groupby(['case','innov_type','nu_true'], dropna=False):
        
        # Read true values directly from stored columns
        omega_true = grp['omega_true'].iloc[0]
        alpha_true = grp['alpha_true'].iloc[0]
        beta_true  = grp['beta_true'].iloc[0]

        for est, cols in [
            ('Sub-Gaussian', ('omega_sub','alpha_sub','beta_sub')),
            ('QMLE',         ('omega_qmle','alpha_qmle','beta_qmle')),
            ('t-MLE',        ('omega_t','alpha_t','beta_t')),
        ]:
            for param, col, true_val in zip(
                ['omega','alpha','beta'], cols,
                [omega_true, alpha_true, beta_true]
            ):
                vals = pd.to_numeric(grp[col], errors='coerce').dropna()
                if vals.empty:
                    continue
                theta_bar = vals.mean()
                bias = theta_bar - true_val
                mcsd = vals.std(ddof=1)
                rmse = np.sqrt(np.mean((vals - true_val)**2))
                rows.append({
                    'case': case, 'innov_type': itype, 'nu_true': nu_true,
                    'estimator': est, 'parameter': param,
                    'bias': bias, 'mcsd': mcsd, 'rmse': rmse,
                    'theta_bar': theta_bar, 'true_val': true_val, 'n_reps': len(vals)
                })
    return pd.DataFrame(rows)

# ── SE ratio table ───────────────────────────────────────────────────────────

def compute_se_ratios(all_results_list):
    """
    For each replication, compute Hessian SE for sub-Gaussian and arch SE
    for QMLE/t-MLE, then take ratio of means across reps.
    """
    rows_se = []
    for rec in all_results_list:
        y = np.asarray(rec['y_series'] if 'y_series' in rec else [], float)
        if len(y) == 0:
            continue

        # Sub-Gaussian SE (only if estimate is valid)
        o_s, a_s, b_s, n_s = rec['omega_sub'], rec['alpha_sub'], rec['beta_sub'], rec['nu_sub']
        if np.all(np.isfinite([o_s, a_s, b_s, n_s])):
            se_sub = sub_se_from_hessian(y, o_s, a_s, b_s, n_s)
        else:
            se_sub = np.full(3, np.nan)

        se_q = arch_se_three(y, dist='normal')
        se_t = arch_se_three(y, dist='t')

        rows_se.append({
            'case': rec['case'], 'innov_type': rec['innov_type'], 'nu_true': rec['nu_true'],
            'se_omega_sub': se_sub[0], 'se_alpha_sub': se_sub[1], 'se_beta_sub': se_sub[2],
            'se_omega_q':   se_q[0],   'se_alpha_q':   se_q[1],   'se_beta_q':   se_q[2],
            'se_omega_t':   se_t[0],   'se_alpha_t':   se_t[1],   'se_beta_t':   se_t[2],
        })

    se_df = pd.DataFrame(rows_se)
    if se_df.empty:
        return se_df, pd.DataFrame()

    mean_se = (se_df.groupby(['case','innov_type','nu_true'], dropna=False)
                    .mean(numeric_only=True).reset_index())

    # ratio = SE_competitor / SE_sub (values < 1 mean sub has larger SE, > 1 means sub is more precise)
    ratio_rows = []
    for _, r in mean_se.iterrows():
        for param in ['omega','alpha','beta']:
            den = r[f'se_{param}_sub']
            ratio_rows.append({
                'case': r['case'], 'innov_type': r['innov_type'], 'nu_true': r['nu_true'],
                'parameter': param,
                'ratio_QMLE_sub': r[f'se_{param}_q']  / den if den > 0 else np.nan,
                'ratio_t_sub':    r[f'se_{param}_t']  / den if den > 0 else np.nan,
            })
    return se_df, pd.DataFrame(ratio_rows)

# ── Coverage ─────────────────────────────────────────────────────────────────

def compute_coverage_sub(all_results_list, alpha=0.05):
    """
    Wald coverage for sub-Gaussian, QMLE, t-MLE.
    For sub-Gaussian, SEs come from numerical Hessian (approximate).
    """
    z = norm_rv.ppf(1 - alpha/2)
    rows = []
    for rec in all_results_list:
        y = np.asarray(rec.get('y_series', []), float)
        if len(y) == 0:
            continue
        case = rec['case']
        omega_true = GARCH_PARAMS[case][0]
        alpha_true = GARCH_PARAMS[case][1]
        beta_true  = GARCH_PARAMS[case][2]
        true_vec = np.array([omega_true, alpha_true, beta_true])

        # Sub-Gaussian
        o_s, a_s, b_s, n_s = rec['omega_sub'], rec['alpha_sub'], rec['beta_sub'], rec['nu_sub']
        if np.all(np.isfinite([o_s, a_s, b_s, n_s])):
            hat = np.array([o_s, a_s, b_s])
            se  = sub_se_from_hessian(y, o_s, a_s, b_s, n_s)
            covers = [float((hat[i]-z*se[i] <= true_vec[i] <= hat[i]+z*se[i]))
                      if np.isfinite(se[i]) else np.nan for i in range(3)]
            ci_lens = [2*z*se[i] if np.isfinite(se[i]) else np.nan for i in range(3)]
            joint = float(all(c == 1.0 for c in covers if np.isfinite(c)))
            rows.append({
                'case': case, 'innov_type': rec['innov_type'], 'nu_true': rec['nu_true'],
                'estimator': 'Sub-Gaussian',
                'cover_omega': covers[0], 'cover_alpha': covers[1], 'cover_beta': covers[2],
                'cover_joint': joint,
                'ci_len_omega': ci_lens[0], 'ci_len_alpha': ci_lens[1], 'ci_len_beta': ci_lens[2]
            })

        # QMLE and t-MLE via arch
        for dist, label in [('normal','QMLE'), ('t','t-MLE')]:
            try:
                model = arch_model(y, mean='zero', vol='GARCH', p=1, q=1, dist=dist)
                res = model.fit(disp='off', show_warning=False, cov_type='robust')
                hat = np.array([res.params['omega'], res.params['alpha[1]'], res.params['beta[1]']])
                se  = res.std_err.reindex(['omega','alpha[1]','beta[1]']).astype(float).to_numpy()
                covers = [float((hat[i]-z*se[i] <= true_vec[i] <= hat[i]+z*se[i]))
                          if np.isfinite(se[i]) else np.nan for i in range(3)]
                ci_lens = [2*z*se[i] if np.isfinite(se[i]) else np.nan for i in range(3)]
                joint = float(all(c == 1.0 for c in covers if np.isfinite(c)))
                rows.append({
                    'case': case, 'innov_type': rec['innov_type'], 'nu_true': rec['nu_true'],
                    'estimator': label,
                    'cover_omega': covers[0], 'cover_alpha': covers[1], 'cover_beta': covers[2],
                    'cover_joint': joint,
                    'ci_len_omega': ci_lens[0], 'ci_len_alpha': ci_lens[1], 'ci_len_beta': ci_lens[2]
                })
            except Exception:
                pass

    cover_long = pd.DataFrame(rows)
    if cover_long.empty:
        return cover_long, pd.DataFrame()

    cover_panel = (cover_long
        .groupby(['case','innov_type','nu_true','estimator'], dropna=False)
        .agg(
            cover_omega=('cover_omega','mean'), cover_alpha=('cover_alpha','mean'),
            cover_beta=('cover_beta','mean'),   cover_joint=('cover_joint','mean'),
            avg_ci_len_omega=('ci_len_omega','mean'),
            avg_ci_len_alpha=('ci_len_alpha','mean'),
            avg_ci_len_beta=('ci_len_beta','mean')
        ).reset_index())
    return cover_long, cover_panel

# ── Run everything ───────────────────────────────────────────────────────────

# Need y_series in all_results — add it to run_one_rep if not already there:
# just add 'y_series': y.tolist() to the returned dict

df_results = pd.DataFrame(all_results)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

print("Computing bias table...")
bias_table = compute_bias_mcsd_rmse(df_results)
bias_table['nu_true'] = bias_table['nu_true'].fillna('normal')
print(bias_table.pivot_table(
    index=['case','innov_type','nu_true','parameter'],
    columns='estimator', values='bias').round(4))

print("\Computing  MCSD table:")
print(bias_table.pivot_table(
    index=['case','innov_type','nu_true','parameter'],
    columns='estimator',
    values='mcsd'
).round(4))

print("\Computing  RMSE table:")
print(bias_table.pivot_table(
    index=['case','innov_type','nu_true','parameter'],
    columns='estimator',
    values='rmse'
).round(4))

print("\nComputing SE ratios (this re-fits each replication - slow)...")
se_df, ratio_df = compute_se_ratios(all_results)
ratio_df['nu_true'] = ratio_df['nu_true'].fillna('normal')
if not ratio_df.empty:
    print(ratio_df.pivot_table(
        index=['case','innov_type','nu_true','parameter'],
        columns=[], values=['ratio_QMLE_sub','ratio_t_sub']).round(4))

print("\nComputing coverage probabilities...")
cover_long, cover_panel = compute_coverage_sub(all_results, alpha=0.05)
cover_long['nu_true'] = cover_long['nu_true'].fillna('normal')
cover_panel['nu_true'] = cover_panel['nu_true'].fillna('normal')
if not cover_panel.empty:
    print("\nMarginal coverage (target=0.95):")
    print(cover_panel.pivot_table(
        index=['case','innov_type','nu_true'],
        columns='estimator',
        values='cover_omega').round(3))
    print("\nJoint coverage:")
    print(cover_panel.pivot_table(
        index=['case','innov_type','nu_true'],
        columns='estimator',
        values='cover_joint').round(3))

Computing bias table...
estimator                              QMLE  Sub-Gaussian   t-MLE
case innov_type   nu_true parameter                              
A    Sub-Gaussian 2.0     alpha      0.0001       -0.0014  0.0005
                          beta      -0.0193       -0.0089 -0.0199
                          omega      0.0188        0.0049  0.0199
                  5.0     alpha      0.0019       -0.0032  0.0022
                          beta      -0.0187       -0.0054 -0.0186
                          omega      0.0170       -0.0003  0.0173
                  8.0     alpha     -0.0008       -0.0057 -0.0001
                          beta      -0.0075       -0.0008 -0.0081
                          omega      0.0080       -0.0048  0.0086
     normal       normal  alpha     -0.0006        0.0758 -0.0004
                          beta      -0.0059       -0.3469 -0.0058
                          omega      0.0082        0.5836  0.0080
     t            5.0     alpha     -0.0011        0

                                     ratio_QMLE_sub  ratio_t_sub
case innov_type   nu_true parameter                             
A    Sub-Gaussian 2.0     alpha           1968.0131    1980.8301
                          beta            3376.4814    3402.9912
                          omega           4475.1483    4541.4494
                  5.0     alpha           1171.4375    1180.2094
                          beta            2518.8018    2528.7963
                          omega           2510.4776    2527.7837
                  8.0     alpha           1224.0749    1242.5208
                          beta            1079.8513    1092.9974
                          omega            741.5566     752.7558
     normal       normal  alpha              3.7954       4.5117
                          beta               4.5292       5.0600
                          omega              2.1511       2.2302
     t            5.0     alpha              4.5942       3.7855
                         

In [7]:
# Representative panel of 3 plots, directly from simulation (correct)
# Rerun once got otuput with +-50% grid search again 

import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('PDF')

np.random.seed(2021)
fix_rep = np.random.randint(0, N)  # randomly pick a rep index

# The three DGPs you want to show
panel_targets = [
    ('Sub-Gaussian', 2.0),
    ('normal', None),
    ('t', 12.0),
]

case = 'A'
omega, alpha, beta_ = GARCH_PARAMS[case]

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

panel_titles = [
    f'Sub-Gaussian Innovations (ν_true = 2.0)',
    f'Gaussian Innovations',
    f'Student-t Innovations (ν_true = 12.0)',
]

for ax, (itype, iparam), title in zip(axes, panel_targets, panel_titles):
    
    # Find the rep from all_results — try fix_rep, fall back if sub estimate failed
    nu_key = iparam  # matches nu_true in your results
    matches = [r for r in all_results 
           if r['case'] == case 
           and r['innov_type'] == itype 
           and r['nu_true'] == nu_key
           and r['rep'] == fix_rep
           and np.all(np.isfinite([r['omega_sub'], r['alpha_sub'], r['beta_sub']]))]
    
    if not matches:
        # fall back: find any rep where sub-Gaussian estimate is valid
        matches = [r for r in all_results 
                   if r['case'] == case 
                   and r['innov_type'] == itype 
                   and r['nu_true'] == nu_key
                   and np.all(np.isfinite([r['omega_sub'], r['alpha_sub'], r['beta_sub']]))]
        if not matches:
            print(f"No valid rep found for {itype} {iparam}, skipping.")
            continue
        rec = matches[0]
    else:
        rec = matches[0]

    used_rep = rec['rep']
    y = np.asarray(rec['y_series'], float)

    # Reconstruct true and estimated volatilities
    sigma2_true = np.asarray(rec['sigma2_true'], float)
    vol_true = np.sqrt(np.maximum(sigma2_true, 1e-12))

    s2_sub = _reconstruct_sigma2(y, rec['omega_sub'], rec['alpha_sub'], rec['beta_sub'])
    s2_q   = _reconstruct_sigma2(y, rec['omega_qmle'], rec['alpha_qmle'], rec['beta_qmle'])
    s2_t   = _reconstruct_sigma2(y, rec['omega_t'],    rec['alpha_t'],    rec['beta_t'])

    vol_sub = np.sqrt(np.maximum(s2_sub, 1e-12))
    vol_q   = np.sqrt(np.maximum(s2_q,   1e-12))
    vol_t   = np.sqrt(np.maximum(s2_t,   1e-12))

    nu_sub_est = rec['nu_sub']
    nu_t_est   = rec['nu_t']

    tgrid = np.arange(len(y))
    ax.plot(tgrid, vol_true, label='True σ_t', color='black', lw=1.5, alpha=0.6)
    ax.plot(tgrid, vol_sub,  label=f'Sub-Gaussian MLE (ν̂={nu_sub_est:.1f})', 
            color='blue', lw=0.9)
    ax.plot(tgrid, vol_q,    label='Gaussian MLE', 
            color='red', lw=0.9, alpha=0.8)
    ax.plot(tgrid, vol_t,    label=f'Student-t MLE (ν̂={nu_t_est:.1f})', 
            color='green', lw=0.9, alpha=0.8)
    ax.set_title(title + f'  [rep={used_rep}]', fontsize=12, fontweight='bold')
    ax.legend(loc='upper right', fontsize=9, frameon=False)
    ax.grid(True, alpha=0.3)

axes[1].set_ylabel('Conditional Volatility σ_t', fontsize=12)
axes[2].set_xlabel('Time (t)', fontsize=12)
plt.tight_layout()
plt.savefig('representative_volatility_panel_caseA_finalsim2.pdf', bbox_inches='tight')
plt.close()
print(f"Saved representative panel (seed=2021, initial rep={fix_rep})")

Saved representative panel (seed=2021, initial rep=85)
